## Problem Statement & Objective

Standard chatbots have two major problems:
1. **No memory** — they forget previous messages in a conversation
2. **Hallucination** — they make up answers not backed by any real source

**Retrieval-Augmented Generation (RAG)** solves both problems:
- Retrieves relevant chunks from a real document corpus before generating a response
- Combined with **conversational memory**, it maintains the flow of multi-turn conversations


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
import os

## Dataset Loading & Preprocessing

I use a custom Wikipedia-style corpus covering 4 topics:
- Artificial Intelligence
- Python Programming  
- Large Language Models
- LangChain Framework

In [12]:
# Custom corpus — 4 Wikipedia-style articles
SAMPLE_CORPUS = [
    {
        "title": "Artificial Intelligence",
        "content": """
        Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to
        natural intelligence displayed by animals including humans. AI research has been defined
        as the field of study of intelligent agents, which refers to any system that perceives
        its environment and takes actions that maximize its chance of achieving its goals.
        Machine learning is a subfield of AI that enables systems to learn from data and improve
        their performance without being explicitly programmed. Deep learning uses neural networks
        with many layers to model complex patterns in data.
        """
    },
    {
        "title": "Python Programming",
        "content": """
        Python is a high-level, general-purpose programming language. Its design philosophy
        emphasizes code readability with significant indentation. Python was created by Guido van
        Rossum and first released in 1991. It has since become one of the most popular languages
        in the world, especially in data science, machine learning, and web development.
        Popular frameworks include Django and Flask for web development, NumPy and Pandas for
        data analysis, and TensorFlow and PyTorch for machine learning.
        """
    },
    {
        "title": "Large Language Models",
        "content": """
        A large language model (LLM) is a language model notable for its ability to achieve
        general-purpose language generation and understanding. LLMs acquire these abilities by
        learning statistical relationships from vast amounts of text during training.
        LLMs use the transformer architecture. GPT by OpenAI and BERT by Google are two
        well-known examples. Retrieval-Augmented Generation (RAG) is a technique that enhances
        LLMs by combining them with a retrieval system to fetch relevant documents and generate
        accurate, grounded responses.
        """
    },
    {
        "title": "LangChain Framework",
        "content": """
        LangChain is an open-source framework for building applications powered by large language
        models. It provides tools for chaining LLM calls, integrating external data sources, and
        managing conversational memory. LangChain supports prompt templates, output parsers,
        vector stores, document loaders, and agents. create_retrieval_chain in LangChain
        allows building chatbots that answer questions from documents while maintaining memory
        of past conversation turns.
        """
    }
]

# Convert to LangChain Document objects
documents = [
    Document(
        page_content=item["content"].strip(),
        metadata={"source": item["title"]}
    )
    for item in SAMPLE_CORPUS
]

print(f"Total documents loaded: {len(documents)}")
for doc in documents:
    print(f"  - {doc.metadata['source']} ({len(doc.page_content)} characters)")

Total documents loaded: 4
  - Artificial Intelligence (622 characters)
  - Python Programming (533 characters)
  - Large Language Models (581 characters)
  - LangChain Framework (499 characters)


In [13]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks after splitting: {len(chunks)}")
print(f"\n--- Sample Chunk ---")
print(f"Source: {chunks[0].metadata['source']}")
print(f"Content: {chunks[0].page_content[:300]}...")

Total chunks after splitting: 7

--- Sample Chunk ---
Source: Artificial Intelligence
Content: Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to
        natural intelligence displayed by animals including humans. AI research has been defined
        as the field of study of intelligent agents, which refers to any system that perceives
        its environment...


## Building the Vector Store

I embed each chunk using `sentence-transformers/all-MiniLM-L6-v2`:
- 384-dimensional dense embeddings
- Runs locally — no API key needed
- Fast and lightweight

Embeddings are stored in a **FAISS** index for fast similarity search.

In [14]:


# Load embedding model — runs locally, no API key needed
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS index from chunks
vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"FAISS index built successfully.")
print(f"Total vectors indexed: {vectorstore.index.ntotal}")
print(f"Embedding dimensions: {vectorstore.index.d}")

C:\Users\Hasnain Arain\AppData\Local\Temp\ipykernel_7868\2560718616.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4303.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index built successfully.
Total vectors indexed: 7
Embedding dimensions: 384


## Testing Retrieval

Before connecting the LLM, i verify that the retriever 
fetches the correct and relevant chunks for a given query

In [15]:
# Test retriever — fetch top 2 chunks for a sample query
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

test_queries = [
    "What is Retrieval-Augmented Generation?",
    "Who created Python?",
    "What is deep learning?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    results = retriever.invoke(query)
    for i, doc in enumerate(results):
        print(f"[{i+1}] Source: {doc.metadata['source']}")
        print(f"     {doc.page_content[:200]}...")


Query: What is Retrieval-Augmented Generation?
--------------------------------------------------
[1] Source: Large Language Models
     A large language model (LLM) is a language model notable for its ability to achieve
        general-purpose language generation and understanding. LLMs acquire these abilities by
        learning stat...
[2] Source: Large Language Models
     LLMs by combining them with a retrieval system to fetch relevant documents and generate
        accurate, grounded responses....

Query: Who created Python?
--------------------------------------------------
[1] Source: Python Programming
     Python is a high-level, general-purpose programming language. Its design philosophy
        emphasizes code readability with significant indentation. Python was created by Guido van
        Rossum and...
[2] Source: LangChain Framework
     LangChain is an open-source framework for building applications powered by large language
        models. It provides tools for chaini

## Model Development — Building the RAG Chain

In [18]:
os.environ["GROQ_API_KEY"] = "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"  

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
    max_tokens=512
)

# Stage 1 — rephrase follow-up questions
rephrase_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("human",
     "Given the conversation history above, rephrase the latest question "
     "into a clear standalone question. Only rephrase, do not answer.")
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, rephrase_prompt
)

# Stage 2 — answer from context only
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful assistant. Answer using ONLY the context below.
If the answer is NOT in the context, say:
"I don't have information about that in my knowledge base."

Context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

qa_chain = create_stuff_documents_chain(llm, qa_prompt)
chain = create_retrieval_chain(history_aware_retriever, qa_chain)

print("RAG chain built successfully.")

RAG chain built successfully.


## Simulated Multi-Turn Conversation

In [19]:
chat_history = []

test_questions = [
    "What is artificial intelligence?",
    "What is machine learning?",           # related follow-up
    "Who created Python?",                 # different topic
    "What is RAG?",                        # from corpus
    "What is the capital of France?"       # out of corpus — should say I don't know
]

for question in test_questions:
    formatted_history = []
    for q, a in chat_history:
        formatted_history.append(HumanMessage(content=q))
        formatted_history.append(AIMessage(content=a))

    result = chain.invoke({
        "input": question,
        "chat_history": formatted_history
    })

    answer = result.get("answer", "No answer generated.")
    chat_history.append((question, answer))

    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)

Q: What is artificial intelligence?
A: Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans.
------------------------------------------------------------
Q: What is machine learning?
A: Machine learning is a subfield of AI that enables systems to learn from data and improve their performance without being explicitly programmed.
------------------------------------------------------------
Q: Who created Python?
A: Python was created by Guido van Rossum.
------------------------------------------------------------
Q: What is RAG?
A: RAG stands for Retrieval-Augmented Generation, a technique that enhances large language models by retrieving relevant information from external sources and using it to generate more accurate and informative responses.
------------------------------------------------------------
Q: What is the capital of France?
A: I don't have information about that in my knowledge base

## Final Summary & Insights

### What i Built
A fully functional context-aware RAG chatbot that:
- Loads and indexes a custom document corpus into FAISS
- Retrieves top-3 relevant chunks per query using cosine similarity
- Rephrases follow-up questions using conversation history (Stage 1)
- Generates grounded answers strictly from retrieved context (Stage 2)

### Deployment
The chatbot is deployed using Streamlit and can be run locally with:
```bash
streamlit run app.py
```
